# Self-sizing TabM: FINAL test batch

k=32, budget 0.5, guard_slack 0.8 (floor = 0.8 x target). 22 test splits, seeds 0-2.
Self-sizing vs matched plain vs default TabM. Resumable: rerun a cell after a disconnect.
Runtime: GPU. Run the cells in order.


## Cell 1: setup
Same setup as Steps 3–6, with the repo pinned to the version this code was tested against.

In [2]:
import os, subprocess, sys
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)

from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/tabm_selfsizing'          # same folder as the Step 3/4 notebook
os.makedirs(ROOT, exist_ok=True)

COMMIT = '28e47ae301c92ec37787dde1ce923a0793f405b4'      # the repo version this code was tested against
if not os.path.exists('/content/tabm'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/yandex-research/tabm', '/content/tabm'], check=True)
    subprocess.run(['git', '-C', '/content/tabm', 'checkout', '-q', COMMIT], check=True)
PAPER = '/content/tabm/paper'
os.chdir(PAPER)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps',
                'delu==0.0.25', 'rtdl_num_embeddings==0.0.11', 'rtdl_revisiting_models==0.0.2'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'loguru', 'tomli', 'tomli-w', 'optuna'], check=True)

util = open('lib/util.py').read()
old = 'return _torch().load(get_checkpoint_path(output), **kwargs)'
new = 'return _torch().load(get_checkpoint_path(output), **{"weights_only": False, **kwargs})'
if old in util:
    open('lib/util.py', 'w').write(util.replace(old, new))
print('Patched:', new in open('lib/util.py').read())

name, memory.total [MiB]
Tesla T4, 15360 MiB

Mounted at /content/drive
Patched: True


## Cell 2: data and dataset lists
Uses the data already in your Drive. Calibration uses only **TUNING** (9 datasets, split 0). **DATASETS** (the 12 test datasets) is never touched here.

In [3]:
import glob
DATA_DRIVE = f'{ROOT}/data'
if not os.path.exists(f'{DATA_DRIVE}/california'):
    os.makedirs(f'{ROOT}/local', exist_ok=True)
    subprocess.run(['wget', '-q', 'https://huggingface.co/datasets/rototoHF/tabm-data/resolve/main/data.tar',
                    '-O', f'{ROOT}/local/tabm-data.tar.gz'], check=True)
    os.makedirs(DATA_DRIVE, exist_ok=True)
    subprocess.run(['tar', '-xf', f'{ROOT}/local/tabm-data.tar.gz', '-C', DATA_DRIVE], check=True)
    inner = [p for p in glob.glob(f'{DATA_DRIVE}/*') if os.path.isdir(p)]
    if len(inner) == 1 and not os.path.exists(f'{inner[0]}/info.json'):
        subprocess.run(f'mv {inner[0]}/* {DATA_DRIVE}/ && rmdir {inner[0]}', shell=True, check=True)
if not os.path.islink(f'{PAPER}/data'):
    if os.path.exists(f'{PAPER}/data'): subprocess.run(['rm', '-rf', f'{PAPER}/data'])
    os.symlink(DATA_DRIVE, f'{PAPER}/data')
os.makedirs(f'{ROOT}/exp_runs', exist_ok=True)
if not os.path.islink(f'{PAPER}/exp/runs'):
    os.symlink(f'{ROOT}/exp_runs', f'{PAPER}/exp/runs')

# The 12 TEST datasets: never used to choose settings
DATASETS = {
    'california': ['california'], 'house': ['house'], 'diamond': ['diamond'],
    'cpu_act': [f'why/regression-num-medium-{i}-cpu_act' for i in range(3)],
    'superconduct': ['why/regression-num-medium-0-superconduct'],
    'churn': ['churn'], 'adult': ['adult'], 'higgs-small': ['higgs-small'],
    'MagicTelescope': [f'why/classif-num-medium-{i}-MagicTelescope' for i in range(3)],
    'phoneme': [f'why/classif-num-medium-{i}-phoneme' for i in range(5)],
    'credit': [f'why/classif-num-medium-{i}-credit' for i in range(2)],
    'otto': ['otto'],
}
# TUNING datasets: other benchmark datasets (split 0), used for checks and calibration only
TUNING = {
    'elevators': ['why/regression-num-medium-0-elevators'],
    'house_sales': ['why/regression-cat-medium-0-house_sales'],
    'wine_quality': ['why/regression-num-medium-0-wine_quality'],
    'MiamiHousing2016': ['why/regression-num-medium-0-MiamiHousing2016'],
    'pol': ['why/regression-num-medium-0-pol'],
    'bank-marketing': ['why/classif-num-medium-0-bank-marketing'],
    'kdd_ipums_la_97-small': ['why/classif-num-medium-0-kdd_ipums_la_97-small'],
    'KDDCup09_upselling': ['why/classif-cat-medium-0-KDDCup09_upselling'],
    'jannis': ['why/classif-num-large-0-jannis'],
}
ALL_SPLITS = {**DATASETS, **TUNING}
VARIANTS = ['tabm', 'tabm-mini-piecewiselinear']
missing = [d for ds in ALL_SPLITS.values() for d in ds
           if not os.path.exists(f'{PAPER}/data/' + d.removeprefix('why/') + '/info.json')]
print('All splits present' if not missing else f'MISSING: {missing}')

All splits present


## Cell 3: create the self-sizing script
Writes `bin/selfsize.py`, including the budget guard (physical pruning never goes below the current target). Runs without `budget_guard` behave as in Step 5.

In [4]:
SRC_DIR = '/content/selfsize_src'
os.makedirs(SRC_DIR, exist_ok=True)
open(f'{SRC_DIR}/selfsize_block.py', 'w').write(r'''
# =====================================================================================
# >>> Self-sizing TabM
#
# One training run that shrinks TabM to a FLOPs budget while it trains:
# - width: one L0 hard-concrete gate per neuron (Louizos et al., 2018), shared by all
#   submodels and all blocks, so a closed gate deletes a whole neuron;
# - depth: one L0 gate per block after the first: x <- x + z * (block(x) - x);
#   z = 1 is the normal TabM block, z = 0 is a pass-through, so the block can be deleted;
# - k: greedy submodel selection on the validation set (the TabM[G] rule), done during
#   training;
# - the budget: a MorphNet-style expected-FLOPs term with a self-adjusting multiplier
#   (Lagrangian dual ascent), and a target lowered linearly from 1.0x to the budget.
# Dead neurons, blocks and submodels are physically removed, together with their Adam
# state, so compute really drops. With [selfsize] enabled = false the script is
# identical to bin/model.py (plus cost accounting).
# =====================================================================================

_HC_GAMMA, _HC_ZETA, _HC_BETA = -0.1, 1.1, 2 / 3


class HardConcreteGate(nn.Module):
    def __init__(self, n: int, init: float) -> None:
        super().__init__()
        self.log_alpha = nn.Parameter(torch.full((n,), init) + 0.01 * torch.randn(n))
        # 'off' -> all ones (warm-up), 'sample' -> stochastic (shrink), 'fixed' -> deterministic
        self.mode = 'off'

    def deterministic(self) -> Tensor:
        return (
            torch.sigmoid(self.log_alpha) * (_HC_ZETA - _HC_GAMMA) + _HC_GAMMA
        ).clamp(0, 1)

    def forward(self) -> Tensor:
        if self.mode == 'off':
            return torch.ones_like(self.log_alpha)
        if self.mode == 'fixed' or not self.training:
            return self.deterministic()
        u = torch.rand_like(self.log_alpha).clamp(1e-6, 1 - 1e-6)
        s = torch.sigmoid((u.log() - (1 - u).log() + self.log_alpha) / _HC_BETA)
        return (s * (_HC_ZETA - _HC_GAMMA) + _HC_GAMMA).clamp(0, 1)

    def prob_open(self) -> Tensor:
        return torch.sigmoid(
            self.log_alpha - _HC_BETA * math.log(-_HC_GAMMA / _HC_ZETA)
        )


class GatedMLP(nn.Module):
    """lib.deep.MLP with a shared width gate and per-block depth gates."""

    def __init__(self, mlp: lib.deep.MLP, gate_init: float) -> None:
        super().__init__()
        assert mlp.output is None
        self.blocks = mlp.blocks
        self.output = None
        self.ckpt = False       # True: recompute the blocks in the backward pass (saves memory)
        self.wgate = HardConcreteGate(self.width, gate_init)
        self.dgate = (
            HardConcreteGate(len(self.blocks) - 1, gate_init)
            if len(self.blocks) > 1
            else None
        )

    @property
    def width(self) -> int:
        return self.blocks[0][0].weight.shape[0]

    def gates(self) -> list[HardConcreteGate]:
        return [g for g in (self.wgate, self.dgate) if g is not None]

    def _run(self, x: Tensor, zw: Tensor, zd: None | Tensor) -> Tensor:
        for i, block in enumerate(self.blocks):
            h = block(x) * zw
            x = h if (i == 0 or zd is None) else x + zd[i - 1] * (h - x)
        return x

    def forward(self, x: Tensor) -> Tensor:
        # gates are sampled here, outside the checkpointed part, so the recomputation sees the same gates
        zw = self.wgate()
        zd = None if self.dgate is None else self.dgate()
        if self.ckpt and self.training and torch.is_grad_enabled():
            return torch.utils.checkpoint.checkpoint(self._run, x, zw, zd, use_reentrant=False)
        return self._run(x, zw, zd)


def _ss_flops(d_in: int, d_out: int, k: int, depth: int, width: int) -> float:
    """Forward FLOPs per object of the backbone + heads (multiply-adds)."""
    return k * (d_in * width + (depth - 1) * width * width + width * d_out)


class SelfSizer:
    def __init__(self, model: 'Model', cfg: None | dict, d_out: int, default_cfg: dict) -> None:
        self.model = model
        self.cfg = dict(cfg or {})
        self.enabled = bool(self.cfg.get('enabled', False))
        self.d_in = model.backbone.blocks[0][0].weight.shape[1]
        self.d_out = d_out
        self.flops_default = _ss_flops(
            self.d_in, d_out, default_cfg['k'], default_cfg['n_blocks'], default_cfg['d_block']
        )
        self.lam = 0.0
        self.train_flops = 0.0
        self.trace: list[dict] = []
        self.k_pruned = False
        if not self.enabled:
            return
        c = self.cfg
        self.budget = float(c['budget'])
        self.warmup = int(c.get('warmup_epochs', 3))
        self.target_end = int(c.get('target_end_epoch', 10))
        self.k_prune_epoch = int(c.get('k_prune_epoch', c.get('freeze_epoch', 15)))  # default: once, at the end of shrinking
        self.prune_every = int(c.get('prune_every', 3))
        self.freeze_epoch = int(c.get('freeze_epoch', 15))
        self.dual_lr = float(c.get('dual_lr', 0.03))
        self.gate_lr = float(c.get('gate_lr', 0.05))
        self.k_min = int(c.get('k_min', 4))
        self.k_keep_frac = float(c.get('k_keep_frac', 0.25))
        self.k_tol = float(c.get('k_tol', 0.002))
        self.w_min = int(c.get('w_min', 16))
        # True: k-pruning may not push the model below the budget (the budget is a target);
        # False: k-pruning is purely accuracy-driven, so the budget is only a cap.
        self.k_respect_budget = bool(c.get('k_respect_budget', False))
        # True: physical pruning never takes the model below the current target (budget guard)
        self.budget_guard = bool(c.get('budget_guard', False))
        self.guard_slack = float(c.get('guard_slack', 1.0))
        model.backbone = GatedMLP(model.backbone, float(c.get('gate_init', 2.5)))
        model.backbone.ckpt = bool(c.get('checkpoint', False))
        self.start_ratio = self.actual_flops() / self.flops_default

    # ---------------------------------------------------------------- size and cost
    @property
    def k(self) -> int:
        return self.model.k

    def config(self) -> tuple[int, int, int]:
        bb = self.model.backbone
        return (self.k, len(bb.blocks), bb.blocks[0][0].weight.shape[0])

    def actual_flops(self) -> float:
        return _ss_flops(self.d_in, self.d_out, *self.config())

    def expected_flops(self) -> Tensor:
        bb = self.model.backbone
        w = bb.wgate.prob_open().sum()
        n_hidden = bb.dgate.prob_open().sum() if bb.dgate is not None else 0.0
        return self.k * (self.d_in * w + n_hidden * w * w + w * self.d_out)

    def target(self, epoch: int) -> float:
        if epoch >= self.target_end:
            return self.budget
        t = (epoch - self.warmup) / max(self.target_end - self.warmup, 1)
        return self.start_ratio + (self.budget - self.start_ratio) * min(max(t, 0.0), 1.0)

    # ---------------------------------------------------------------- phases
    def shrinking(self, epoch: int) -> bool:
        return self.enabled and self.warmup < epoch <= self.freeze_epoch

    def size_fixed(self, epoch: int) -> bool:
        """Best-model tracking and early stopping only once the size can no longer change."""
        return (not self.enabled) or epoch > self.freeze_epoch

    def custom_groups(self) -> list[dict]:
        if not self.enabled:
            return []
        params = [p for g in self.model.backbone.gates() for p in g.parameters()]
        return [{'params': params, 'lr': 0.0, 'weight_decay': 0.0, 'selfsize_gates': True}]

    def start_epoch(self, epoch: int, optimizer: torch.optim.Optimizer) -> None:
        if not self.enabled:
            return
        on = self.shrinking(epoch)
        for g in self.model.backbone.gates():
            g.mode = 'sample' if on else ('fixed' if epoch > self.freeze_epoch else 'off')
        for group in optimizer.param_groups:
            if group.get('selfsize_gates'):
                group['lr'] = self.gate_lr if on else 0.0
        self._epoch = epoch

    def add_penalty(self, loss: Tensor) -> Tensor:
        if not (self.enabled and self.shrinking(self._epoch)):
            return loss
        gap = self.expected_flops() / self.flops_default - self.target(self._epoch)
        self._gap = gap.item()
        return loss + self.lam * gap

    def after_step(self, n_objects: int) -> None:
        # n_objects = objects seen by EACH submodel in this step; forward + backward ~ 3x forward
        self.train_flops += 3 * n_objects * self.actual_flops()
        if self.enabled and self.shrinking(self._epoch):
            self.lam = max(0.0, self.lam + self.dual_lr * self._gap)

    def end_epoch(self, epoch: int, optimizer, head_val: np.ndarray, y_val: np.ndarray,
                  task_type: str, val_score: float) -> None:
        if self.enabled:
            # width/depth first, then k, so a budget-aware k step sees the final width and depth
            if self.warmup < epoch <= self.freeze_epoch and (
                (epoch - self.warmup) % self.prune_every == 0 or epoch == self.freeze_epoch
            ):
                guard = self.target(epoch) * self.guard_slack if self.budget_guard else None
                self._prune_width(optimizer, guard)
                self._prune_depth(optimizer, guard)
            if epoch == self.k_prune_epoch and not self.k_pruned:
                if len(self.model.backbone.blocks) and head_val.shape[1] == self.k:
                    self._prune_k(optimizer, self._greedy_k(head_val, y_val, task_type))
                self.k_pruned = True
            if epoch == self.freeze_epoch:
                for g in self.model.backbone.gates():
                    g.mode = 'fixed'
                    g.log_alpha.requires_grad_(False)
        k, depth, width = self.config()
        self.trace.append({
            'epoch': epoch, 'k': k, 'depth': depth, 'width': width,
            'flops_x_default': self.actual_flops() / self.flops_default,
            'expected_flops_x_default': (
                float(self.expected_flops().item()) / self.flops_default if self.enabled else None
            ),
            'target': self.target(epoch) if self.enabled else None,
            'lambda': self.lam, 'val_score': val_score,
        })

    # ---------------------------------------------------------------- pruning
    @staticmethod
    def _replace(module, name, fn, optimizer) -> None:
        old = getattr(module, name)
        new = nn.Parameter(fn(old.data).contiguous(), requires_grad=old.requires_grad)
        setattr(module, name, new)
        st = optimizer.state.pop(old, None)
        if st is not None:
            optimizer.state[new] = {
                kk: (fn(v).contiguous() if torch.is_tensor(v) and v.shape == old.shape else v)
                for kk, v in st.items()
            }
        for g in optimizer.param_groups:
            g['params'] = [new if p is old else p for p in g['params']]

    def _sync_optimizer(self, optimizer) -> None:
        alive = {id(p) for p in self.model.parameters()}
        for g in optimizer.param_groups:
            g['params'] = [p for p in g['params'] if id(p) in alive]
        for p in list(optimizer.state):
            if id(p) not in alive:
                del optimizer.state[p]

    @torch.no_grad()
    def _prune_width(self, optimizer, floor_ratio: None | float = None) -> None:
        bb = self.model.backbone
        z = bb.wgate.deterministic()
        keep = torch.where(z > 0)[0]
        if len(keep) < self.w_min:
            keep = z.topk(self.w_min).indices.sort().values
        if floor_ratio is not None:
            # budget guard: never keep fewer neurons than the current target needs
            k, depth = self.model.k, len(bb.blocks)
            need = next(
                (w for w in range(1, bb.width + 1)
                 if _ss_flops(self.d_in, self.d_out, k, depth, w) >= floor_ratio * self.flops_default),
                bb.width,
            )
            if len(keep) < need:
                keep = z.topk(need).indices.sort().values
        if len(keep) == bb.width:
            return
        sel = lambda dim: (lambda t: t.index_select(dim, keep))
        for i, block in enumerate(bb.blocks):
            lin = block[0]
            self._replace(lin, 'weight', sel(0), optimizer)              # output neurons
            if i > 0:
                self._replace(lin, 'weight', sel(1), optimizer)          # input neurons
                if getattr(lin, 'r', None) is not None:
                    self._replace(lin, 'r', sel(1), optimizer)
            if getattr(lin, 's', None) is not None:
                self._replace(lin, 's', sel(1), optimizer)
            if lin.bias is not None:
                self._replace(lin, 'bias', sel(lin.bias.ndim - 1), optimizer)
            if hasattr(lin, 'out_features'):
                lin.out_features = len(keep)
                if i > 0:
                    lin.in_features = len(keep)
        self._replace(self.model.output, 'weight', sel(1), optimizer)
        self._replace(bb.wgate, 'log_alpha', sel(0), optimizer)

    @torch.no_grad()
    def _prune_depth(self, optimizer, floor_ratio: None | float = None) -> None:
        bb = self.model.backbone
        if bb.dgate is None:
            return
        keep = torch.where(bb.dgate.deterministic() > 0)[0]
        if len(keep) == len(bb.blocks) - 1:
            return
        if floor_ratio is not None and _ss_flops(
                self.d_in, self.d_out, self.model.k, 1 + len(keep), bb.width) < floor_ratio * self.flops_default:
            return                                   # budget guard: this would go below the target
        bb.blocks = nn.ModuleList([bb.blocks[0]] + [bb.blocks[i + 1] for i in keep.tolist()])
        if len(keep) == 0:
            bb.dgate = None
        else:
            self._replace(bb.dgate, 'log_alpha', lambda t: t.index_select(0, keep), optimizer)
        self._sync_optimizer(optimizer)

    @torch.no_grad()
    def _prune_k(self, optimizer, keep: list[int]) -> None:
        if len(keep) == self.k:
            return
        idx = torch.as_tensor(sorted(keep), device=self.model.output.weight.device)
        sel = lambda t: t.index_select(0, idx)
        if self.model.minimal_ensemble_adapter is not None:              # TabM-mini
            self._replace(self.model.minimal_ensemble_adapter, 'weight', sel, optimizer)
        for block in self.model.backbone.blocks:                         # TabM
            lin = block[0]
            for nm in ('r', 's'):
                if getattr(lin, nm, None) is not None:
                    self._replace(lin, nm, sel, optimizer)
            if lin.bias is not None and lin.bias.ndim == 2:
                self._replace(lin, 'bias', sel, optimizer)
            if hasattr(lin, 'k'):
                lin.k = len(idx)
        self._replace(self.model.output, 'weight', sel, optimizer)
        self._replace(self.model.output, 'bias', sel, optimizer)
        self.model.k = len(idx)

    def _greedy_k(self, head_val: np.ndarray, y_val: np.ndarray, task_type: str) -> list[int]:
        """TabM[G]: greedily add the submodel that most improves the COLLECTIVE validation
        loss; keep the smallest subset within k_tol of the full ensemble's loss."""
        P = torch.as_tensor(head_val, dtype=torch.float64)          # (N, k) or (N, k, C)
        y = torch.as_tensor(y_val)
        if task_type == 'regression':
            loss = lambda p: ((p - y.double()[:, None]) ** 2).mean(0)
        elif task_type == 'binclass':
            yy = y.double()[:, None]
            loss = lambda p: -(yy * p.clamp(1e-9, 1).log()
                               + (1 - yy) * (1 - p).clamp(1e-9, 1).log()).mean(0)
        else:
            yi = y.long()
            loss = lambda p: -p.gather(-1, yi.view(-1, 1, 1).expand(-1, p.shape[1], 1)
                                       ).squeeze(-1).clamp_min(1e-9).log().mean(0)
        k = P.shape[1]
        full = loss(P.mean(1, keepdim=True))[0].item()
        chosen, remaining, run_sum, curve = [], list(range(k)), None, []
        for n in range(k):
            cand = P[:, remaining] if run_sum is None else (run_sum.unsqueeze(1) + P[:, remaining]) / (n + 1)
            ls = loss(cand)
            j = int(ls.argmin())
            curve.append(ls[j].item())
            chosen.append(remaining.pop(j))
            run_sum = P[:, chosen[-1]] if run_sum is None else run_sum + P[:, chosen[-1]]
        ok = [i + 1 for i, l in enumerate(curve) if l <= full * (1 + self.k_tol)]
        m = ok[0] if ok else k
        m = max(m, math.ceil(k * self.k_keep_frac), self.k_min)
        if self.k_respect_budget:
            ratio = self.actual_flops() / self.flops_default      # flops scale linearly with k
            m = max(m, math.ceil(k * self.budget / ratio - 1e-9)) if ratio > self.budget else k
        return chosen[: min(m, k)]

    def summary(self) -> dict:
        k, depth, width = self.config()
        return {
            'enabled': self.enabled, 'final_k': k, 'final_depth': depth, 'final_width': width,
            'final_flops_x_default': self.actual_flops() / self.flops_default,
            'train_flops': self.train_flops, 'flops_default': self.flops_default,
            'trace': self.trace,
        }

# <<< Self-sizing TabM
# =====================================================================================

''')
open(f'{SRC_DIR}/make_selfsize.py', 'w').write(r'''"""Create paper/bin/selfsize.py from paper/bin/model.py: same script + self-sizing hooks."""
import sys
from pathlib import Path
paper = Path(sys.argv[1])
src = (paper / 'bin/model.py').read_text()
block = (Path(__file__).parent / 'selfsize_block.py').read_text()

def sub(old, new, count=1):
    global src
    assert src.count(old) == count, (src.count(old), old)
    src = src.replace(old, new)

# 1. the self-sizing classes, before the Config type
sub('class Config(TypedDict):', block + '\n\nclass Config(TypedDict):')
sub('    parameter_statistics: NotRequired[bool]\n',
    '    parameter_statistics: NotRequired[bool]\n    selfsize: NotRequired[KWArgs]\n')
# 2. wrap the model right after it is built
sub("""    report['n_parameters'] = lib.deep.get_n_parameters(model)""",
"""    _defaults = config.get('selfsize', {}).get('default_size', {'k': 32, 'n_blocks': 3, 'd_block': 512})
    sizer = SelfSizer(
        model, config.get('selfsize'),
        d_out=1 if dataset.task.is_regression else dataset.task.compute_n_classes(),
        default_cfg=_defaults,
    )
    report['n_parameters'] = lib.deep.get_n_parameters(model)""")
# 3. gates get their own optimizer group
sub("""**config['optimizer'], params=lib.deep.make_parameter_groups(model)""",
    """**config['optimizer'],
        params=lib.deep.make_parameter_groups(model, custom_groups=sizer.custom_groups())""")
# 4. per-submodel batches follow the CURRENT k
sub("""                    (config['model']['k'], dataset.size('train')),""",
    """                    (model.k, dataset.size('train')),""")
# 5. epoch start, penalty, dual step
sub("""        model.train()
        epoch_losses = []""",
"""        epoch = step // epoch_size + 1
        sizer.start_epoch(epoch, optimizer)
        model.train()
        epoch_losses = []""")
sub("""            loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])""",
"""            loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])
            loss = sizer.add_penalty(loss)""")
sub("""            step += 1
            epoch_losses.append(loss.detach())""",
"""            step += 1
            epoch_losses.append(loss.detach())
            sizer.after_step(len(batch_idx) if share_training_batches else len(batch_idx) // model.k)""")
# 6. epoch end: pruning on validation head predictions; no best-model tracking while the size can change
sub("""        metrics, predictions, _, eval_batch_size = evaluate(
            ['val', 'test'], eval_batch_size
        )""",
"""        metrics, predictions, _head_preds, eval_batch_size = evaluate(
            ['val', 'test'], eval_batch_size
        )
        _y_val = dataset.data['y']['val'].cpu().numpy()
        if regression_label_stats is not None:
            _y_val = _y_val * regression_label_stats.std + regression_label_stats.mean
        sizer.end_epoch(
            epoch, optimizer, _head_preds['val'], _y_val,
            dataset.task.type_.value, metrics['val']['score'],
        )""")
sub("""        if (
            'metrics' not in report
            or metrics['val']['score'] > report['metrics']['val']['score']
        ):""",
"""        if not sizer.size_fixed(epoch):
            print()
            continue
        if (
            'metrics' not in report
            or metrics['val']['score'] > report['metrics']['val']['score']
        ):""")
# 7. cost report
sub("""    report['time'] = str(timer)
""",
"""    report['time'] = str(timer)
    report['epochs_trained'] = step // epoch_size
    report['peak_memory_mb'] = (
        torch.cuda.max_memory_allocated() / 2**20 if torch.cuda.is_available() else None
    )
    report['selfsize'] = sizer.summary()
    report['n_parameters_final'] = lib.deep.get_n_parameters(model)
""")
(paper / 'bin/selfsize.py').write_text(src)
print('wrote', paper / 'bin/selfsize.py')
''')
subprocess.run([sys.executable, f'{SRC_DIR}/make_selfsize.py', PAPER], check=True)
print('bin/selfsize.py ready:', os.path.exists(f'{PAPER}/bin/selfsize.py'))

bin/selfsize.py ready: True


## Cell 4: runner and summaries
`SS` is the Step 5 setting. The calibration overrides budget, budget_guard and k.

In [5]:
import json, time, copy, tomli, tomli_w
import numpy as np, pandas as pd
from pathlib import Path

# Self-sizing settings (the ones Step 6 calibrates). Budget = final forward FLOPs as a multiple
# of default TabM's (same variant, same dataset). Schedule in epochs.
SS = dict(
    budget=0.5,
    warmup_epochs=3,        # plain training, gates off
    target_end_epoch=10,    # target lowered linearly from 1.0x to the budget by this epoch
    prune_every=3,          # physically delete closed neurons/blocks every N epochs
    freeze_epoch=15,        # last shrinking epoch; k is pruned here too, then the size is frozen
    dual_lr=0.03,           # how fast the lambda multiplier adapts
    gate_lr=0.05, gate_init=2.5,
    k_min=4, k_keep_frac=0.25, k_tol=0.002, w_min=16,
    k_respect_budget=False, # True: k-pruning may not push the model below the budget
)

def make_config(variant, split, seed, ss=None):
    """Package-default hyperparameters (no per-dataset tuning) on the paper's protocol."""
    cfg = tomli.loads((Path('exp') / variant / split / '0-evaluation' / f'{seed}.toml').read_text())
    has_emb = 'num_embeddings' in cfg['model']
    cfg['amp'] = False
    cfg['head_selection'] = False
    cfg['model']['backbone'].update(n_blocks=2 if has_emb else 3, d_block=512, dropout=0.1)
    cfg['optimizer'].update(lr=2e-3, weight_decay=3e-4)
    if has_emb:
        cfg['model']['num_embeddings']['d_embedding'] = 16
        cfg['bins']['n_bins'] = 48
    default_size = {'k': 32, 'n_blocks': 2 if has_emb else 3, 'd_block': 512}
    cfg['selfsize'] = ({'enabled': False, 'default_size': default_size} if ss is None
                       else {'enabled': True, **ss, 'default_size': default_size})
    return cfg

def train(variant, split, seed, tag, ss=None, overrides=None):
    out = Path('exp/runs') / tag / variant / split
    report = out / str(seed) / 'report.json'
    if report.exists():
        return json.loads(report.read_text())
    out.mkdir(parents=True, exist_ok=True)
    cfg = make_config(variant, split, seed, ss)
    for k, v in (overrides or {}).items():
        cfg[k] = v
    (out / f'{seed}.toml').write_text(tomli_w.dumps(cfg))
    t0 = time.time()
    r = subprocess.run([sys.executable, 'bin/selfsize.py', str(out / f'{seed}.toml'), '--force'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(f'failed: {tag} {variant} {split} {seed}')
    rep = json.loads(report.read_text()); s = rep['selfsize']
    print(f'  {tag:<12} {variant:<26} {split:<44} seed={seed} score={rep["metrics"]["test"]["score"]:.4f} '
          f'size=({s["final_k"]},{s["final_depth"]},{s["final_width"]}) {time.time() - t0:.0f}s')
    return rep

def run(names, tag, seeds, ss=None, table=None, overrides=None):
    table = table or ALL_SPLITS
    for name in names:
        for split in table[name]:
            for variant in VARIANTS:
                for s in seeds:
                    train(variant, split, s, tag, ss, overrides)

def secs(t):
    h, m, s = t.split(':'); return int(h) * 3600 + int(m) * 60 + float(s)

def collect(tag, names, table=None):
    table = table or ALL_SPLITS
    rows = []
    for name in names:
        for split in table[name]:
            for variant in VARIANTS:
                for rep in (Path('exp/runs') / tag / variant / split).glob('*/report.json'):
                    if not rep.parent.name.isdigit():
                        continue
                    r = json.loads(rep.read_text()); s = r['selfsize']
                    rows.append({'dataset': name, 'split': split, 'variant': variant, 'seed': r['config']['seed'],
                                 'score': r['metrics']['test']['score'], 'time_s': secs(r['time']),
                                 'peak_mem_mb': r.get('peak_memory_mb'), 'epochs': r['epochs_trained'],
                                 'train_flops': s['train_flops'], 'final_flops_x': s['final_flops_x_default'],
                                 'size': (s['final_k'], s['final_depth'], s['final_width'])})
    return pd.DataFrame(rows)

def compare(tag, names, base='off', table=None):
    """Self-sizing (tag) vs the same model with self-sizing off (base), per dataset and variant."""
    a, b = collect(base, names, table), collect(tag, names, table)
    out = []
    for (name, variant), g in b.groupby(['dataset', 'variant']):
        h = a[(a.dataset == name) & (a.variant == variant)]
        if h.empty: continue
        sign = -1 if g.score.mean() < 0 else 1                      # RMSE is stored as -RMSE
        out.append({'dataset': name, 'variant': variant, 'runs': len(g),
                    'base_score': round(sign * h.score.mean(), 4), 'base_std': round(h.score.std(), 4),
                    'ss_score': round(sign * g.score.mean(), 4), 'ss_std': round(g.score.std(), 4),
                    'rel_change_%': round((g.score.mean() - h.score.mean()) / abs(h.score.mean()) * 100, 2),
                    'final_flops_x': round(g.final_flops_x.mean(), 3),
                    'train_flops_x': round(g.train_flops.mean() / h.train_flops.mean(), 2),
                    'time_x': round(g.time_s.mean() / h.time_s.mean(), 2),
                    'mem_x': (round(g.peak_mem_mb.mean() / h.peak_mem_mb.mean(), 2)
                              if g.peak_mem_mb.notna().all() and h.peak_mem_mb.notna().all() else None),
                    'final_sizes': sorted(set(g['size']))})
    df = pd.DataFrame(out)
    df.to_csv(f'{ROOT}/compare_{tag}.csv', index=False)
    return df

## Cell 5: freeze record and final settings
Appends the revised freeze to ROOT/FREEZE.txt (never overwrites it). Set STEP4_TAG to the folder Step 4 used for default TabM; leave it '' to train default TabM here.

In [6]:
import datetime
FINAL = {'k': 32, 'budget': 0.5, 'budget_guard': True, 'k_keep_frac': 1.0, 'k_respect_budget': False, 'guard_slack': 0.8}
TEST_SEEDS = [0, 1, 2]
TEST_VARIANT = 'tabm'
TEST_ITEMS = [(n, s) for n in DATASETS for s in DATASETS[n]]
assert len(TEST_ITEMS) == 21, len(TEST_ITEMS)
STEP4_TAG = ''   # e.g. 'default' if Step 4 reports are in exp/runs/default; '' trains default here

SS_FINAL = {**SS, 'budget': FINAL['budget'], 'budget_guard': FINAL['budget_guard'],
            'k_keep_frac': FINAL['k_keep_frac'], 'k_respect_budget': FINAL['k_respect_budget'],
            'guard_slack': FINAL['guard_slack']}

def secs(t):
    h, m, s = t.split(':'); return int(h) * 3600 + int(m) * 60 + float(s)

def cfg_with_k(variant, split, seed, k, ss=None):
    cfg = make_config(variant, split, seed, ss)
    cfg['model']['k'] = int(k)
    return cfg

def plain_cfg(split, seed, width):
    cfg = cfg_with_k(TEST_VARIANT, split, seed, FINAL['k'], None)
    cfg['model']['backbone']['d_block'] = int(width)
    return cfg

freeze = Path(f'{ROOT}/FREEZE.txt')
with open(freeze, 'a') as f:
    f.write(f"\nFinal test batch started: {datetime.datetime.now().isoformat(timespec='minutes')} (local time)\n"
            f"Setting: {FINAL}\n"
            "Replaces the 0.5x freeze (guard_slack 1.0) after the tuning check: flex, seed 1, mean vs default +0.65%, FLOPs 0.42x.\n"
            "Win rule: mean change vs default TabM >= -1% across the 22 test splits.\n")
print(freeze.read_text()[-600:])
print(f"Test runs: {len(TEST_ITEMS)} splits x {len(TEST_SEEDS)} seeds x 2 = {len(TEST_ITEMS)*len(TEST_SEEDS)*2} (+ default TabM if not from Step 4)")

ect_budget': False, 'guard_slack': 0.8}
Replaces the 0.5x freeze (guard_slack 1.0) after the tuning check: flex, seed 1, mean vs default +0.65%, FLOPs 0.42x.
Win rule: mean change vs default TabM >= -1% across the 22 test splits.

Final test batch started: 2026-11-17T15:03 (local time)
Setting: {'k': 32, 'budget': 0.5, 'budget_guard': True, 'k_keep_frac': 1.0, 'k_respect_budget': False, 'guard_slack': 0.8}
Replaces the 0.5x freeze (guard_slack 1.0) after the tuning check: flex, seed 1, mean vs default +0.65%, FLOPs 0.42x.
Win rule: mean change vs default TabM >= -1% across the 22 test splits.

Test runs: 21 splits x 3 seeds x 2 = 126 (+ default TabM if not from Step 4)


## Cell 6: final test runs (self-sizing and matched plain)
132 runs. Resumable.

In [8]:
def run_cfg(cfg, out, seed, tag):
    report = out / str(seed) / 'report.json'
    if report.exists():
        return json.loads(report.read_text())
    out.mkdir(parents=True, exist_ok=True)
    (out / f'{seed}.toml').write_text(tomli_w.dumps(cfg))
    t0 = time.time()
    r = subprocess.run([sys.executable, 'bin/selfsize.py', str(out / f'{seed}.toml'), '--force'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(f'failed: {tag}')
    rep = json.loads(report.read_text())
    print(f'  {tag:<34} score={rep["metrics"]["test"]["score"]:.4f} '
          f'size=({rep["selfsize"]["final_k"]},{rep["selfsize"]["final_depth"]},{rep["selfsize"]["final_width"]}) {time.time()-t0:.0f}s')
    return rep

for n, s in TEST_ITEMS:
    for seed in TEST_SEEDS:
        g = run_cfg(cfg_with_k(TEST_VARIANT, s, seed, FINAL['k'], SS_FINAL),
                    Path('exp/runs') / 'final-self' / TEST_VARIANT / s, seed, f'self {n} {s} s{seed}')
        w = g['selfsize']['final_width']
        run_cfg(plain_cfg(s, seed, w),
                Path('exp/runs') / 'final-plain' / TEST_VARIANT / s, seed, f'plain {n} w{w} s{seed}')

## Cell 7: default TabM on the same splits and seeds
Uses Step 4 reports if STEP4_TAG is set and they exist; otherwise trains default TabM (k=32, width 512, depth 3).

In [9]:
def default_report(s, seed):
    if STEP4_TAG:
        p = Path('exp/runs') / STEP4_TAG / TEST_VARIANT / s / str(seed) / 'report.json'
        if p.exists():
            return json.loads(p.read_text()), 'step4'
    rep = run_cfg(cfg_with_k(TEST_VARIANT, s, seed, FINAL['k'], None),
                  Path('exp/runs') / 'final-default' / TEST_VARIANT / s, seed, f'default {s} s{seed}')
    return rep, 'trained'

sources = []
for n, s in TEST_ITEMS:
    for seed in TEST_SEEDS:
        _, src_ = default_report(s, seed)
        sources.append(src_)
print('default TabM sources:', {x: sources.count(x) for x in set(sources)})

  default diamond s2                 score=-0.1312 size=(32,3,512) 284s
  default why/regression-num-medium-0-cpu_act s0 score=-2.0681 size=(32,3,512) 61s
  default why/regression-num-medium-0-cpu_act s1 score=-2.0928 size=(32,3,512) 48s
  default why/regression-num-medium-0-cpu_act s2 score=-2.0728 size=(32,3,512) 53s
  default why/regression-num-medium-1-cpu_act s0 score=-2.2093 size=(32,3,512) 65s
  default why/regression-num-medium-1-cpu_act s1 score=-2.2338 size=(32,3,512) 65s
  default why/regression-num-medium-1-cpu_act s2 score=-2.2211 size=(32,3,512) 53s
  default why/regression-num-medium-2-cpu_act s0 score=-2.1692 size=(32,3,512) 66s
  default why/regression-num-medium-2-cpu_act s1 score=-2.2030 size=(32,3,512) 47s
  default why/regression-num-medium-2-cpu_act s2 score=-2.1923 size=(32,3,512) 56s
  default why/regression-num-medium-0-superconduct s0 score=-10.2666 size=(32,3,512) 182s
  default why/regression-num-medium-0-superconduct s1 score=-10.2587 size=(32,3,512) 257s
 

In [ ]:
# DIAGNOSTIC (post-hoc, not the frozen test): physical pruning only once, at freeze_epoch
import json, time, subprocess, sys
import numpy as np, pandas as pd, tomli_w
from pathlib import Path

DIAG_ITEMS = [('cpu_act', s) for s in DATASETS['cpu_act']] + [('superconduct', s) for s in DATASETS['superconduct']]
DIAG_SEEDS = [0, 1, 2]
SS_DIAG = {**SS_FINAL, 'prune_every': 1000}   # prune only at freeze_epoch

def cfg_with_k(variant, split, seed, k, ss=None):
    cfg = make_config(variant, split, seed, ss)
    cfg['model']['k'] = int(k)
    return cfg

def plain_cfg(split, seed, width):
    cfg = cfg_with_k('tabm', split, seed, 32, None)
    cfg['model']['backbone']['d_block'] = int(width)
    return cfg

def run_cfg(cfg, out, seed, tag):
    report = out / str(seed) / 'report.json'
    if report.exists():
        return json.loads(report.read_text())
    out.mkdir(parents=True, exist_ok=True)
    (out / f'{seed}.toml').write_text(tomli_w.dumps(cfg))
    t0 = time.time()
    r = subprocess.run([sys.executable, 'bin/selfsize.py', str(out / f'{seed}.toml'), '--force'],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-3000:]); raise RuntimeError(f'failed: {tag}')
    rep = json.loads(report.read_text())
    print(f'  {tag:<32} score={rep["metrics"]["test"]["score"]:.4f} '
          f'size=({rep["selfsize"]["final_k"]},{rep["selfsize"]["final_depth"]},{rep["selfsize"]["final_width"]}) {time.time()-t0:.0f}s')
    return rep

for n, s in DIAG_ITEMS:
    for seed in DIAG_SEEDS:
        g = run_cfg(cfg_with_k('tabm', s, seed, FINAL['k'], SS_DIAG),
                    Path('exp/runs') / 'diag-self' / 'tabm' / s, seed, f'diag self {n} s{seed}')
        w = g['selfsize']['final_width']
        run_cfg(plain_cfg(s, seed, w),
                Path('exp/runs') / 'diag-plain' / 'tabm' / s, seed, f'diag plain {n} w{w} s{seed}')

## Cell 8: analysis and win check
Win rule (set before the test): mean change vs default TabM >= -1% across the 22 splits. Reports the full table.

In [10]:
from scipy.stats import wilcoxon
WIN_MARGIN = -1.0
rows = []
for n, s in TEST_ITEMS:
    for seed in TEST_SEEDS:
        d, _ = default_report(s, seed)
        g = json.loads((Path('exp/runs') / 'final-self' / TEST_VARIANT / s / str(seed) / 'report.json').read_text())
        p = json.loads((Path('exp/runs') / 'final-plain' / TEST_VARIANT / s / str(seed) / 'report.json').read_text())
        ds, gs, ps = d['metrics']['test']['score'], g['metrics']['test']['score'], p['metrics']['test']['score']
        rows.append({'dataset': n, 'split': s, 'seed': seed,
                     'vs_default_%': (gs - ds) / abs(ds) * 100,
                     'vs_plain_%': (gs - ps) / abs(ps) * 100,
                     'final_flops_x': g['selfsize']['final_flops_x_default'],
                     'final_depth': g['selfsize']['final_depth'],
                     'final_width': g['selfsize']['final_width'],
                     'train_flops_x': g['selfsize']['train_flops'] / p['selfsize']['train_flops'],
                     'time_x': secs(g['time']) / secs(p['time']),
                     'mem_x': (g['peak_memory_mb'] / p['peak_memory_mb']) if g.get('peak_memory_mb') and p.get('peak_memory_mb') else None})
df = pd.DataFrame(rows)
df.to_csv(f'{ROOT}/final_rows.csv', index=False)
per_split = df.groupby(['dataset', 'split'])[['vs_default_%', 'vs_plain_%']].mean()
per_split.round(3).to_csv(f'{ROOT}/final_per_split.csv')
print(per_split.round(2).to_string())

def wil(x):
    try: return wilcoxon(x).pvalue
    except ValueError: return float('nan')

d_def = per_split['vs_default_%'].values
d_pl = per_split['vs_plain_%'].values
m = d_def.mean()
print()
print(f'vs DEFAULT TabM: mean {m:+.2f}%  median {np.median(d_def):+.2f}%  wins(>+0.5%) {(d_def>0.5).sum()}  losses(<-0.5%) {(d_def<-0.5).sum()}  Wilcoxon p {wil(d_def):.3f}')
print(f'  WIN RULE (mean >= {WIN_MARGIN}%): {"WIN" if m >= WIN_MARGIN else "NOT A WIN"}')
print(f'vs matched PLAIN: mean {d_pl.mean():+.2f}%  median {np.median(d_pl):+.2f}%  wins {(d_pl>0.5).sum()}  losses {(d_pl<-0.5).sum()}  Wilcoxon p {wil(d_pl):.3f}')
print(f'final FLOPs x default: mean {df.final_flops_x.mean():.3f}  (min {df.final_flops_x.min():.3f}, max {df.final_flops_x.max():.3f})')
print(f'final depth counts: {df.final_depth.value_counts().to_dict()}')
print(f'final width: mean {df.final_width.mean():.0f}')
print(f'training FLOPs x plain: {df.train_flops_x.mean():.2f}')
print(f'time x plain: {df.time_x.mean():.2f}')
print(f'memory x plain: {df.mem_x.dropna().mean():.2f}' if df.mem_x.notna().any() else 'memory: n/a')

                                                         vs_default_%  vs_plain_%
dataset        split                                                             
MagicTelescope why/classif-num-medium-0-MagicTelescope           0.32        0.23
               why/classif-num-medium-1-MagicTelescope           0.07        0.18
               why/classif-num-medium-2-MagicTelescope           0.30        0.18
adult          adult                                            -0.11       -0.01
california     california                                        1.26        0.47
churn          churn                                             0.08        0.06
cpu_act        why/regression-num-medium-0-cpu_act              -5.10       -5.45
               why/regression-num-medium-1-cpu_act              -1.84       -3.54
               why/regression-num-medium-2-cpu_act              -3.26       -4.12
credit         why/classif-num-medium-0-credit                  -0.19       -0.15
               w

In [11]:
# Plain vs default TabM: the cost of the smaller size with no self-sizing involved
pl_rows = []
for n, s in TEST_ITEMS:
    for seed in TEST_SEEDS:
        d, _ = default_report(s, seed)
        p = json.loads((Path('exp/runs') / 'final-plain' / TEST_VARIANT / s / str(seed) / 'report.json').read_text())
        ds, ps = d['metrics']['test']['score'], p['metrics']['test']['score']
        pl_rows.append({'dataset': n, 'split': s, 'seed': seed,
                        'plain_vs_default_%': (ps - ds) / abs(ds) * 100})
pl = pd.DataFrame(pl_rows)
pl.to_csv(f'{ROOT}/plain_vs_default_rows.csv', index=False)
pl_split = pl.groupby(['dataset', 'split'])['plain_vs_default_%'].mean()
print(pl_split.round(2).to_string())

d_pl = pl_split.values
print()
print(f'plain vs DEFAULT TabM: mean {d_pl.mean():+.2f}%  median {np.median(d_pl):+.2f}%  '
      f'wins(>+0.5%) {(d_pl>0.5).sum()}  losses(<-0.5%) {(d_pl<-0.5).sum()}')

dataset         split                                   
MagicTelescope  why/classif-num-medium-0-MagicTelescope     0.08
                why/classif-num-medium-1-MagicTelescope    -0.11
                why/classif-num-medium-2-MagicTelescope     0.13
adult           adult                                      -0.10
california      california                                  0.79
churn           churn                                       0.02
cpu_act         why/regression-num-medium-0-cpu_act         0.33
                why/regression-num-medium-1-cpu_act         1.64
                why/regression-num-medium-2-cpu_act         0.82
credit          why/classif-num-medium-0-credit            -0.05
                why/classif-num-medium-1-credit             0.27
diamond         diamond                                     0.38
higgs-small     higgs-small                                 0.06
house           house                                      -0.41
otto            otto             

In [13]:
def sc(r): return r['metrics']['test']['score']
def ld(tag, s, seed):
    return json.loads((Path('exp/runs') / tag / 'tabm' / s / str(seed) / 'report.json').read_text())

rows = []
for n, s in DIAG_ITEMS:
    for seed in DIAG_SEEDS:
        fs, fp = ld('final-self', s, seed), ld('final-plain', s, seed)
        dg, dp = ld('diag-self', s, seed), ld('diag-plain', s, seed)
        d, _ = default_report(s, seed)
        rows.append({'dataset': n, 'split': s, 'seed': seed,
                     'frozen_vs_plain_%': (sc(fs) - sc(fp)) / abs(sc(fp)) * 100,
                     'diag_vs_plain_%': (sc(dg) - sc(dp)) / abs(sc(dp)) * 100,
                     'diag_vs_default_%': (sc(dg) - sc(d)) / abs(sc(d)) * 100,
                     'diag_width': dg['selfsize']['final_width']})
df = pd.DataFrame(rows)
df.to_csv(f'{ROOT}/diag_prune_end.csv', index=False)
print(df.groupby(['dataset', 'split'])[['frozen_vs_plain_%', 'diag_vs_plain_%', 'diag_vs_default_%']].mean().round(2).to_string())
print(f"\nmean diag vs plain: {df['diag_vs_plain_%'].mean():+.2f}%   frozen vs plain: {df['frozen_vs_plain_%'].mean():+.2f}%")
print(f"mean diag vs default: {df['diag_vs_default_%'].mean():+.2f}%")

NameError: name 'DIAG_ITEMS' is not defined